<a href="https://colab.research.google.com/github/EllenKAlves/ProjetoAurora/blob/main/script_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Leitura dos Dados

In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/sample_data/telemetria_marte.csv')
df.head(10)


# Estatística Descritiva

In [11]:
df.describe()

,temperatura_interna_c,temperatura_externa_c,integridade_estrutural,nivel_energia_pct,pressao_tanques_bar
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,22.563840,-180.397040,0.982000,87.536760,27.483320
std,1.429875,17.643183,0.133084,7.206797,4.362603
min,20.010000,-209.930000,0.000000,75.170000,19.860000
25%,21.285000,-196.290000,1.000000,81.312500,23.692500
50%,22.615000,-180.475000,1.000000,87.490000,27.560000
75%,23.860000,-164.987500,1.000000,93.860000,31.230000
max,24.980000,-150.080000,1.000000,99.960000,34.980000


# Verificações

In [16]:
limites = {
    "temp_interna": (15, 30),        # faixa segura
    "temp_externa": (-220, -140),    # criogênico
    "energia_min": 20,               # %
    "pressao_min": 5                 # bar (adaptado)
}

# Criando Alertas

In [18]:
def analisar_linha(linha):
    alertas = []

    # Temperatura interna
    if not (limites["temp_interna"][0] <= linha["temperatura_interna_c"] <= limites["temp_interna"][1]):
        alertas.append("TEMP_INTERNA_FORA")

    # Temperatura externa
    if not (limites["temp_externa"][0] <= linha["temperatura_externa_c"] <= limites["temp_externa"][1]):
        alertas.append("TEMP_EXTERNA_FORA")

    # Energia
    if linha["nivel_energia_pct"] < limites["energia_min"]:
        alertas.append("ENERGIA_BAIXA")

    # Pressão
    if linha["pressao_tanques_bar"] < limites["pressao_min"]:
        alertas.append("PRESSAO_BAIXA")

    # Integridade
    if linha["integridade_estrutural"] == 0:
        alertas.append("FALHA_ESTRUTURAL")

    # Módulos críticos
    if linha["status_navegacao"] != "OK":
        alertas.append("FALHA_NAVEGACAO")

    return alertas

# Processamento

In [19]:
def processar(df):
    total_alertas = 0
    eventos_criticos = 0

    for i, linha in df.iterrows():
        alertas = analisar_linha(linha)

        if alertas:
            total_alertas += len(alertas)

            # Se tiver falha estrutural ou navegação → crítico
            if "FALHA_ESTRUTURAL" in alertas or "FALHA_NAVEGACAO" in alertas:
                eventos_criticos += 1

            print(f"[{linha['timestamp']}] ALERTAS: {', '.join(alertas)}")

    return total_alertas, eventos_criticos

In [25]:
def main():
    print("🔍 Iniciando análise de telemetria...\n")

    print(f"Total de registros: {len(df)}\n")

    total_alertas, eventos_criticos = processar(df)

    print("\n==============================")
    print("📊 RESULTADO FINAL")
    print("==============================")
    print(f"Total de alertas: {total_alertas}")
    print(f"Eventos críticos: {eventos_criticos}")

    if eventos_criticos > 0:
        print("⚠️ MISSÃO EM RISCO")
    else:
        print("✅ MISSÃO ESTÁVEL")

main()

🔍 Iniciando análise de telemetria...

Total de registros: 500

[2026-03-25T00:31:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T01:30:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T01:44:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T02:43:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T02:47:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T03:59:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T05:48:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T06:25:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO
[2026-03-25T08:01:00Z] ALERTAS: FALHA_ESTRUTURAL, FALHA_NAVEGACAO

📊 RESULTADO FINAL
Total de alertas: 18
Eventos críticos: 9
⚠️ MISSÃO EM RISCO


# Resolvendo os Alertas


In [36]:
# Adiciona uma coluna no dataframe, indicando se a linha apresenta um alerta ou não
def marcar_alertas(df):
    df["tem_alerta"] = df.apply(lambda linha: len(analisar_linha(linha)) > 0, axis=1)
    return df

In [ ]:
## Criando um novo dataframe apenas para demonstrar as linhas com alerta
df = marcar_alertas(df)

linhas_com_alerta = df[df["tem_alerta"] == True]

print("Total de linhas com alerta:", len(linhas_com_alerta))
linhas_com_alerta

In [ ]:
## Removendo as linhas com alerta do dataframe original
df_limpo = df[df["tem_alerta"] == False]
## Visualizando o novo dataframe
df_limpo.head(10)

# Rodando o teste novamente

In [ ]:
print("\n🚀 Reexecutando análise com dados limpos...\n")

total_alertas, eventos_criticos = processar(df_limpo)

print("\n==============================")
print("📊 RESULTADO APÓS LIMPEZA")
print("==============================")
print(f"Total de alertas: {total_alertas}")
print(f"Eventos críticos: {eventos_criticos}")

In [51]:
def main():
    print("🔍 Iniciando análise de telemetria...\n")

    print(f"Total de registros: {len(df_limpo)}\n")

    total_alertas, eventos_criticos = processar(df_limpo)

    print("\n==============================")
    print("📊 RESULTADO FINAL")
    print("==============================")
    print(f"Total de alertas: {total_alertas}")
    print(f"Eventos críticos: {eventos_criticos}")

    if eventos_criticos > 0:
        print("⚠️ MISSÃO EM RISCO")
    else:
        print("✅ MISSÃO ESTÁVEL")

main()

🔍 Iniciando análise de telemetria...

Total de registros: 491


📊 RESULTADO FINAL
Total de alertas: 0
Eventos críticos: 0
✅ MISSÃO ESTÁVEL
